In [9]:
!pip install deep-sort-realtime
!pip install torchreid
!pip install opencv-python
!pip install motmetrics

/usr/bin/pip:6: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import load_entry_point
     |████████████████████████████████| 8.4 MB 4.7 MB/s eta 0:00:01
     |████████████████████████████████| 34.5 MB 6.0 MB/s eta 0:00:01
ERROR: jax 0.4.13 requires ml-dtypes>=0.1.0, which is not installed.
ERROR: jax 0.4.13 requires opt-einsum, which is not installed.
/usr/bin/pip:6: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import load_entry_point
     |████████████████████████████████| 92 kB 1.9 MB/s eta 0:00:01
  Created wheel for torchreid: filename=torchreid-0.2.5-py3-none-any.whl size=144325 sha256=ae826671a28a3fb0c8c8789f2edcf86f8049c40894afa83ae16e1172e41430d5
  Stored in directory: /home/hardik/.cache/pip/wheels/0c/76/88/0f1a27b220aab822eb61ca8c23089607bc683f2fe402f9a8db
Successfully built torchreid
/usr

In [12]:
!pip install trackeval

/usr/bin/pip:6: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import load_entry_point
     |████████████████████████████████| 207 kB 4.2 MB/s eta 0:00:01
     |████████████████████████████████| 439 kB 68.5 MB/s eta 0:00:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [15]:
import torch
from deep_sort_realtime.deepsort_tracker import DeepSort
import torchreid
import cv2
import numpy as np
import trackeval
print("All imports successful")
print(f"Torch version:     {torch.__version__}")
print(f"CUDA available:    {torch.cuda.is_available()}")
print(f"OpenCV version:    {cv2.__version__}")
print(f"TrackEval version: {trackeval.__version__}")

All imports successful
Torch version:     2.4.1+cu121
CUDA available:    True
OpenCV version:    4.13.0
TrackEval version: 1.3.0


In [14]:
!pip install tensorboard

/usr/bin/pip:6: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import load_entry_point
     |████████████████████████████████| 5.5 MB 4.7 MB/s eta 0:00:01
     |████████████████████████████████| 135 kB 79.5 MB/s eta 0:00:01
     |████████████████████████████████| 6.0 MB 25 kB/s s eta 0:00:01
     |████████████████████████████████| 240 kB 43 kB/s s eta 0:00:01
     |████████████████████████████████| 106 kB 76.9 MB/s eta 0:00:01
     |████████████████████████████████| 6.6 MB 76.1 MB/s eta 0:00:01
     |████████████████████████████████| 227 kB 5.6 MB/s eta 0:00:01
     |████████████████████████████████| 4.5 MB 6.0 MB/s eta 0:00:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PAT

loading model for experiment 5 with the MC Dropout

In [16]:
import torch
import torch.nn as nn
from torchvision.models.detection import fasterrcnn_resnet50_fpn
import torchvision
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class FastRCNNPredictorWithDropout(nn.Module):
    def __init__(self, in_channels, num_classes, dropout_p=0.2):
        super().__init__()
        self.dropout   = nn.Dropout(p=dropout_p)
        self.cls_score = nn.Linear(in_channels, num_classes)
        self.bbox_pred = nn.Linear(in_channels, num_classes * 4)

    def forward(self, x):
        x           = self.dropout(x)
        scores      = self.cls_score(x)
        bbox_deltas = self.bbox_pred(x)
        return scores, bbox_deltas

original_fastrcnn_loss = torchvision.models.detection.roi_heads.fastrcnn_loss
def fastrcnn_loss_with_smoothing(class_logits, box_regression, labels, regression_targets):
    num_classes = class_logits.shape[1]
    labels_cat  = torch.cat(labels, dim=0)
    smoothed    = torch.full_like(class_logits, 0.1 / num_classes)
    smoothed.scatter_(1, labels_cat.unsqueeze(1), 1.0 - 0.1 + 0.1 / num_classes)
    cls_loss = (
        F.cross_entropy(class_logits, labels_cat) * 0.9 +
        (-smoothed * F.log_softmax(class_logits, dim=1)).sum(dim=1).mean() * 0.1
    )
    _, box_loss = original_fastrcnn_loss(
        class_logits, box_regression, labels, regression_targets
    )
    return cls_loss, box_loss

torchvision.models.detection.roi_heads.fastrcnn_loss = fastrcnn_loss_with_smoothing


model = fasterrcnn_resnet50_fpn(pretrained=False)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictorWithDropout(
    in_features, num_classes=2, dropout_p=0.2
)
model.load_state_dict(torch.load(
    "model_weights/best_model_exp5.pth",  
    map_location=device
))
model.to(device)

model.train()
for module in model.modules():
    if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
        module.eval()   

print("Exp 5 model loaded with MC Dropout ready")

/home/hardik/.local/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hardik/.local/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/tmp/ipykernel_1149790/219656831.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This

Exp 5 model loaded with MC Dropout ready


In [17]:
    import numpy as np
    
    def mc_dropout_inference(model, frame, device, 
                              score_threshold=0.7, 
                              n_runs=10):
    
        img_tensor = torch.tensor(frame).permute(2, 0, 1).float() / 255.0
        img_tensor = img_tensor.to(device)
    
        all_boxes  = []
        all_scores = []
    
        with torch.no_grad():
            for _ in range(n_runs):
                outputs = model([img_tensor])
                boxes   = outputs[0]['boxes'].cpu().numpy()
                scores  = outputs[0]['scores'].cpu().numpy()
                keep    = scores >= score_threshold
                boxes   = boxes[keep]
                scores  = scores[keep]
    
                all_boxes.append(boxes)
                all_scores.append(scores)
    
        if len(all_boxes[0]) == 0:
            return np.array([]), np.array([]), np.array([])
    
        ref_boxes      = all_boxes[0]
        score_matrix   = np.zeros((len(ref_boxes), n_runs))
        score_matrix[:, 0] = all_scores[0]
    
        for run_idx in range(1, n_runs):
            run_boxes  = all_boxes[run_idx]
            run_scores = all_scores[run_idx]
    
            if len(run_boxes) == 0:
                continue
    
            for ref_idx, ref_box in enumerate(ref_boxes):
                best_iou   = 0
                best_score = 0
                for run_box, run_score in zip(run_boxes, run_scores):
                    iou = compute_iou(ref_box, run_box)
                    if iou > best_iou:
                        best_iou   = iou
                        best_score = run_score
                if best_iou > 0.5:
                    score_matrix[ref_idx, run_idx] = best_score
    
        mean_scores  = score_matrix.mean(axis=1)      
        uncertainty  = score_matrix.var(axis=1)         
    
        return ref_boxes, mean_scores, uncertainty
    
    
    def compute_iou(box1, box2):
        """Compute IoU between two boxes [x1,y1,x2,y2]"""
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
    
        intersection = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
        area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
        union = area1 + area2 - intersection
    
        return intersection / (union + 1e-6)
    
    print("MC Dropout inference function ready")

MC Dropout inference function ready


In [22]:
import cv2
test_img_path = "MOT17/test/MOT17-01-DPM/img1/000001.jpg"  
frame = cv2.imread(test_img_path)
frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
frame = cv2.resize(frame, (640, 640))

boxes, mean_scores, uncertainty = mc_dropout_inference(
    model, frame, device,
    score_threshold=0.7,
    n_runs=10
)

print(f"Detections found:  {len(boxes)}")
print(f"Mean scores:       {mean_scores.round(3)}")
print(f"Uncertainty:       {uncertainty.round(4)}")
print(f"High uncertainty:  {(uncertainty > 0.01).sum()} detections")

Detections found:  12
Mean scores:       [0.998 0.996 0.995 0.994 0.993 0.968 0.932 0.894 0.932 0.464 0.604 0.072]
Uncertainty:       [0.     0.     0.     0.     0.     0.0003 0.0007 0.0007 0.0006 0.1441
 0.0921 0.0473]
High uncertainty:  3 detections


In [20]:
def enable_dropout(model):
    """Enable dropout layers only, keep everything else in eval mode"""
    model.eval()  
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()  

enable_dropout(model)
print("MC Dropout enabled — model in eval mode with dropout active")

MC Dropout enabled — model in eval mode with dropout active


In [21]:
def mc_dropout_inference(model, frame, device,
                          score_threshold=0.7,
                          n_runs=10):
    img_tensor = torch.tensor(frame).permute(2, 0, 1).float() / 255.0
    img_tensor = img_tensor.to(device)

    all_boxes  = []
    all_scores = []

    with torch.no_grad():  
        for _ in range(n_runs):
            outputs = model([img_tensor])  
            boxes   = outputs[0]['boxes'].cpu().numpy()
            scores  = outputs[0]['scores'].cpu().numpy()

            keep   = scores >= score_threshold
            boxes  = boxes[keep]
            scores = scores[keep]

            all_boxes.append(boxes)
            all_scores.append(scores)

    if len(all_boxes[0]) == 0:
        return np.array([]), np.array([]), np.array([])

    ref_boxes    = all_boxes[0]
    score_matrix = np.zeros((len(ref_boxes), n_runs))
    score_matrix[:, 0] = all_scores[0]

    for run_idx in range(1, n_runs):
        run_boxes  = all_boxes[run_idx]
        run_scores = all_scores[run_idx]
        if len(run_boxes) == 0:
            continue
        for ref_idx, ref_box in enumerate(ref_boxes):
            best_iou   = 0
            best_score = 0
            for run_box, run_score in zip(run_boxes, run_scores):
                iou = compute_iou(ref_box, run_box)
                if iou > best_iou:
                    best_iou   = iou
                    best_score = run_score
            if best_iou > 0.5:
                score_matrix[ref_idx, run_idx] = best_score

    mean_scores = score_matrix.mean(axis=1)
    uncertainty = score_matrix.var(axis=1)

    return ref_boxes, mean_scores, uncertainty

print("MC Dropout inference function ready")

MC Dropout inference function ready


Setting the ReID module

In [23]:
import torchreid

reid_model = torchreid.models.build_model(
    name='osnet_x0_25',      
    num_classes=1000,        
    pretrained=True
)
reid_model.eval()
reid_model = reid_model.to(device)
print("OSNet ReID model loaded!")

Downloading...
From: https://drive.google.com/uc?id=1rb8UN5ZzPKRc_xvtHlyDh-cSz88YX9hs
To: /home/hardik/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth
100%|██████████| 2.97M/2.97M [00:00<00:00, 8.56MB/s]

Successfully loaded imagenet pretrained weights from "/home/hardik/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"
OSNet ReID model loaded!



/home/hardik/.local/lib/python3.8/site-packages/torchreid/reid/models/osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(cached_file)


In [24]:
import torchvision.transforms as T


reid_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((256, 128)),      # standard ReID input size
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

def extract_reid_features(reid_model, frame, boxes, device):

    embeddings = []

    if len(boxes) == 0:
        return embeddings

    for box in boxes:
        x1, y1, x2, y2 = map(int, box)

        # clamp to frame boundaries
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(frame.shape[1], x2)
        y2 = min(frame.shape[0], y2)

        # skip invalid crops
        if x2 <= x1 or y2 <= y1:
            embeddings.append(np.zeros(512))
            continue

        crop = frame[y1:y2, x1:x2]   

        # skip tiny crops
        if crop.shape[0] < 10 or crop.shape[1] < 10:
            embeddings.append(np.zeros(512))
            continue

        crop_tensor = reid_transform(crop).unsqueeze(0).to(device)

        with torch.no_grad():
            embedding = reid_model(crop_tensor)
            embedding = embedding.cpu().numpy().flatten()

        embeddings.append(embedding)

    return embeddings

print("ReID feature extraction function ready")

ReID feature extraction function ready


In [25]:
embeddings = extract_reid_features(reid_model, frame, boxes, device)

print(f"Number of embeddings:  {len(embeddings)}")
print(f"Embedding dimension:   {len(embeddings[0])}")
print(f"Sample embedding norm: {np.linalg.norm(embeddings[0]):.4f}")

Number of embeddings:  12
Embedding dimension:   512
Sample embedding norm: 24.8150


Setting the deepsort tracker

In [26]:
from deep_sort_realtime.deepsort_tracker import DeepSort

tracker = DeepSort(
    max_age=30,          
    n_init=3,            
    nn_budget=100,      
    max_cosine_distance=0.3  
)
print("DeepSORT tracker ready")

DeepSORT tracker ready


/home/hardik/.local/lib/python3.8/site-packages/deep_sort_realtime/embedder/embedder_pytorch.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(

In [27]:
def uncertainty_gate(boxes, mean_scores, uncertainty, embeddings,
                     uncertainty_threshold=0.05):

    filtered_boxes       = []
    filtered_scores      = []
    filtered_embeddings  = []
    uncertainty_weights  = []

    for i, (box, score, unc, emb) in enumerate(
            zip(boxes, mean_scores, uncertainty, embeddings)):

        if unc > uncertainty_threshold:
            print(f"  Detection {i} filtered — uncertainty {unc:.4f} > {uncertainty_threshold}")
            continue

    
        weight = 1.0 / (1.0 + unc * 100)
        weighted_score = score * weight

        filtered_boxes.append(box)
        filtered_scores.append(weighted_score)
        filtered_embeddings.append(emb)
        uncertainty_weights.append(weight)

    return filtered_boxes, filtered_scores, filtered_embeddings, uncertainty_weights

print("Uncertainty gating function ready")

Uncertainty gating function ready


In [28]:
def format_for_deepsort(boxes, scores, embeddings, frame_h, frame_w):

    detections = []

    for box, score, emb in zip(boxes, scores, embeddings):
        x1, y1, x2, y2 = box
        w = x2 - x1
        h = y2 - y1

        if w <= 0 or h <= 0:
            continue

        detections.append((
            [x1, y1, w, h],
            float(score),
            np.array(emb)
        ))

    return detections

print("DeepSORT formatting function ready")

DeepSORT formatting function ready


In [29]:

print("=== Testing full pipeline on single frame ===\n")


boxes, mean_scores, uncertainty = mc_dropout_inference(
    model, frame, device,
    score_threshold=0.7,
    n_runs=10
)
print(f"Step 1 — Detections:     {len(boxes)}")
print(f"         Uncertainties:  {uncertainty.round(4)}")

embeddings = extract_reid_features(reid_model, frame, boxes, device)
print(f"\nStep 2 — Embeddings:     {len(embeddings)} x {len(embeddings[0])}d")

filtered_boxes, filtered_scores, filtered_embeddings, weights = uncertainty_gate(
    boxes, mean_scores, uncertainty, embeddings,
    uncertainty_threshold=0.05
)
print(f"\nStep 3 — After gating:   {len(filtered_boxes)} detections remaining")
print(f"         Filtered out:   {len(boxes) - len(filtered_boxes)} high uncertainty")

h, w = frame.shape[:2]
deepsort_dets = format_for_deepsort(
    filtered_boxes, filtered_scores, filtered_embeddings, h, w
)

tracks = tracker.update_tracks(deepsort_dets, frame=frame)
confirmed_tracks = [t for t in tracks if t.is_confirmed()]
print(f"\nStep 4 — Active tracks:  {len(confirmed_tracks)}")
for track in confirmed_tracks:
    print(f"         Track ID {track.track_id}: box {[int(x) for x in track.to_ltrb()]}")

=== Testing full pipeline on single frame ===

Step 1 — Detections:     10
         Uncertainties:  [0.     0.     0.     0.     0.     0.0007 0.0004 0.0022 0.0008 0.1151]

Step 2 — Embeddings:     10 x 512d
  Detection 9 filtered — uncertainty 0.1151 > 0.05

Step 3 — After gating:   9 detections remaining
         Filtered out:   1 high uncertainty

Step 4 — Active tracks:  0


In [30]:
import json
from pathlib import Path

print("Folder structure:")
for seq in sorted(Path("val_data_withgt").iterdir()):
    if seq.is_dir():
        imgs = list((seq / "img1").glob("*.jpg"))
        print(f"  {seq.name}  |  images: {len(imgs)}")

with open("val_data_withgt/annotations.json") as f:
    ann = json.load(f)

keys = list(ann.keys())[:3]
print(f"\nTotal entries in annotations.json: {len(ann)}")
print(f"Sample keys: {keys}")
print(f"Sample value: {ann[keys[0]]}")

Folder structure:
  MOT17-02-DPM  |  images: 114
  MOT17-04-DPM  |  images: 199
  MOT17-05-DPM  |  images: 154
  MOT17-09-DPM  |  images: 103
  MOT17-10-DPM  |  images: 138
  MOT17-11-DPM  |  images: 197
  MOT17-13-DPM  |  images: 159

Total entries in annotations.json: 1064
Sample keys: ['MOT17-04-DPM/000506.jpg', 'MOT17-04-DPM/000792.jpg', 'MOT17-09-DPM/000114.jpg']
Sample value: [[1554.0, 597.0, 1628.0, 839.0], [449.0, 10.0, 523.0, 190.0], [822.0, 609.0, 897.0, 865.0], [1724.0, 459.0, 1798.0, 665.0], [1029.0, 933.0, 1116.0, 1187.0], [639.0, 757.0, 709.0, 1007.0], [789.0, 145.0, 847.0, 316.0], [999.0, 0, 1051.0, 149.0], [932.0, 0, 987.0, 160.0], [561.0, 64.0, 600.0, 219.0], [876.0, 0, 928.0, 123.0], [150.0, 328.0, 211.0, 519.0], [102.0, 296.0, 147.0, 509.0], [930.0, 689.0, 1053.0, 944.0], [1023.0, 705.0, 1099.0, 939.0], [756.0, 101.0, 805.0, 233.0], [721.0, 92.0, 785.0, 259.0], [709.0, 0, 757.0, 130.0], [550.0, 0, 591.0, 132.0], [326.0, 177.0, 375.0, 345.0], [292.0, 785.0, 366.0, 104

In [34]:
import os
import glob
import pandas as pd
from pathlib import Path

train_root = "train-main"  
val_root   = "val_data_withgt"

for seq_dir in sorted(Path(val_root).iterdir()):
    if not seq_dir.is_dir():
        continue

    seq_name = seq_dir.name   
    orig_gt  = Path(train_root) / seq_name / "gt" / "gt.txt"

    if not orig_gt.exists():
        print(f"[SKIP] No original GT found for {seq_name}")
        continue

    df = pd.read_csv(orig_gt, header=None,
                     names=['frame','id','x','y','w','h','conf','class','vis'])

    df = df[(df['class'] == 1) & (df['vis'] >= 0.1)]

    
    val_imgs   = sorted((seq_dir / "img1").glob("*.jpg"))
    val_frames = set(int(p.stem) for p in val_imgs)

    print(f"{seq_name}  |  val frames: {len(val_frames)}")

    df_val = df[df['frame'].isin(val_frames)]

    gt_out_dir = seq_dir / "gt"
    gt_out_dir.mkdir(exist_ok=True)
    df_val.to_csv(gt_out_dir / "gt.txt", header=False, index=False)

    print(f"  GT rows saved: {len(df_val)}")

print("\n All GT files written")

MOT17-02-DPM  |  val frames: 114
  GT rows saved: 2115
MOT17-04-DPM  |  val frames: 199
  GT rows saved: 8532
MOT17-05-DPM  |  val frames: 154
  GT rows saved: 963
MOT17-09-DPM  |  val frames: 103
  GT rows saved: 809
MOT17-10-DPM  |  val frames: 138
  GT rows saved: 2333
MOT17-11-DPM  |  val frames: 197
  GT rows saved: 1806
MOT17-13-DPM  |  val frames: 159
  GT rows saved: 2231

 All GT files written


In [33]:
import subprocess
subprocess.run(["python3", "-m", "pip", "install", "pandas==1.5.3", "--user"], check=True)

  Using cached pandas-1.5.3-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
Using cached pandas-1.5.3-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.2 MB)
Using cached pytz-2026.1.post1-py2.py3-none-any.whl (510 kB)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
notebook 7.4.7 requires jupyterlab<4.5,>=4.4.9, but you have jupyterlab 4.3.8 which is incompatible.


CompletedProcess(args=['python3', '-m', 'pip', 'install', 'pandas==1.5.3', '--user'], returncode=0)

In [35]:
from pathlib import Path

val_root = Path("val_data_withgt")

print("Final val structure:\n")
for seq_dir in sorted(val_root.iterdir()):
    if not seq_dir.is_dir():
        continue
    
    imgs    = list((seq_dir / "img1").glob("*.jpg"))
    gt_file = seq_dir / "gt" / "gt.txt"

    first_line = ""
    if gt_file.exists():
        with open(gt_file) as f:
            first_line = f.readline().strip()
    
    print(f"  {seq_dir.name}")
    print(f"    images:     {len(imgs)}")
    print(f"    gt exists:  {gt_file.exists()}")
    print(f"    first line: {first_line}\n")

Final val structure:

  MOT17-02-DPM
    images:     114
    gt exists:  True
    first line: 7,2,1374,415,162,384,1,1,1.0

  MOT17-04-DPM
    images:     199
    gt exists:  True
    first line: 2,1,1362,568,103,241,1,1,0.86173

  MOT17-05-DPM
    images:     154
    gt exists:  True
    first line: 1,1,17,150,77,191,1,1,1.0

  MOT17-09-DPM
    images:     103
    gt exists:  True
    first line: 2,1,262,449,102,263,1,1,1.0

  MOT17-10-DPM
    images:     138
    gt exists:  True
    first line: 2,1,1366,394,75,229,1,1,1.0

  MOT17-11-DPM
    images:     197
    gt exists:  True
    first line: 7,1,888,146,187,628,1,1,1.0

  MOT17-13-DPM
    images:     159
    gt exists:  True
    first line: 4,2,1383,513,33,99,1,1,0.61912



In [36]:
import os
import cv2
import glob
import numpy as np
from pathlib import Path
from tqdm import tqdm
from deep_sort_realtime.deepsort_tracker import DeepSort

val_root  = Path("val_data_withgt")
pred_root = Path("predictions")   
pred_root.mkdir(exist_ok=True)

for seq_dir in sorted(val_root.iterdir()):
    if not seq_dir.is_dir():
        continue

    seq_name  = seq_dir.name
    img_files = sorted((seq_dir / "img1").glob("*.jpg"))
    print(f"\n[{seq_name}] — {len(img_files)} frames")

    tracker = DeepSort(
        max_age=30,
        n_init=3,
        nn_budget=100,
        max_cosine_distance=0.3
    )

    pred_lines = []

    for img_path in tqdm(img_files, desc=f"  Tracking"):

        frame_num = int(img_path.stem)

        frame_bgr    = cv2.imread(str(img_path))
        frame_rgb    = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = frame_rgb.shape[:2]
        frame_resized  = cv2.resize(frame_rgb, (640, 640))
        scale_x = orig_w / 640
        scale_y = orig_h / 640

        boxes, mean_scores, uncertainty = mc_dropout_inference(
            model, frame_resized, device,
            score_threshold=0.7,
            n_runs=10
        )

        if len(boxes) > 0:
            boxes[:, [0, 2]] *= scale_x
            boxes[:, [1, 3]] *= scale_y

        embeddings = extract_reid_features(reid_model, frame_rgb, boxes, device)

        filtered_boxes, filtered_scores, filtered_embeddings, _ = uncertainty_gate(
            boxes, mean_scores, uncertainty, embeddings,
            uncertainty_threshold=0.05
        )

        deepsort_dets = format_for_deepsort(
            filtered_boxes, filtered_scores, filtered_embeddings,
            orig_h, orig_w
        )
        tracks = tracker.update_tracks(deepsort_dets, frame=frame_rgb)

        for t in tracks:
            if not t.is_confirmed():
                continue
            x1, y1, x2, y2 = t.to_ltrb()
            w = x2 - x1
            h = y2 - y1
            
            pred_lines.append(
                f"{frame_num},{t.track_id},{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},1,-1,-1,-1\n"
            )

    pred_file = pred_root / f"{seq_name}.txt"
    with open(pred_file, "w") as f:
        f.writelines(pred_lines)

    print(f"  Predictions saved: {len(pred_lines)} rows → {pred_file}")

print("\n All sequences tracked and saved")

/home/hardik/.local/lib/python3.8/site-packages/deep_sort_realtime/embedder/embedder_pytorch.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(


[MOT17-02-DPM] — 114 frames


  Tracking:   2%|▏         | 2/114 [00:02<02:03,  1.10s/it]

  Detection 22 filtered — uncertainty 0.1356 > 0.05


  Tracking:   3%|▎         | 3/114 [00:03<01:58,  1.07s/it]

...

  Detection 7 filtered — uncertainty 0.1056 > 0.05


  Tracking:  99%|█████████▊| 157/159 [02:17<00:01,  1.50it/s]

  Detection 6 filtered — uncertainty 0.0728 > 0.05


  Tracking: 100%|██████████| 159/159 [02:18<00:00,  1.15it/s]

  Predictions saved: 7083 rows → predictions/MOT17-13-DPM.txt

 All sequences tracked and saved


In [37]:
from pathlib import Path

pred_root = Path("predictions")

print("Saved prediction files:\n")
for f in sorted(pred_root.glob("*.txt")):
    with open(f) as file:
        lines = file.readlines()
    print(f"  {f.name}  |  rows: {len(lines)}")

Saved prediction files:

  MOT17-02-DPM.txt  |  rows: 3938
  MOT17-04-DPM.txt  |  rows: 9414
  MOT17-05-DPM.txt  |  rows: 2450
  MOT17-09-DPM.txt  |  rows: 1458
  MOT17-10-DPM.txt  |  rows: 5759
  MOT17-11-DPM.txt  |  rows: 3610
  MOT17-13-DPM.txt  |  rows: 7083


In [39]:
import os
import shutil
from pathlib import Path

base = Path("trackeval_data")

for seq_dir in sorted(Path("val_data_withgt").iterdir()):
    if not seq_dir.is_dir():
        continue

    seq_name = seq_dir.name

    # GT folders
    gt_dst = base / "gt" / "mot_challenge" / "MOT17-val" / seq_name / "gt"
    gt_dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(seq_dir / "gt" / "gt.txt", gt_dst / "gt.txt")

    # seqinfo.ini — TrackEval needs basic sequence info
    imgs      = sorted((seq_dir / "img1").glob("*.jpg"))
    img       = __import__('cv2').imread(str(imgs[0]))
    h, w      = img.shape[:2]
    seqinfo   = (
        f"[Sequence]\n"
        f"name={seq_name}\n"
        f"imDir=img1\n"
        f"frameRate=30\n"
        f"seqLen={len(imgs)}\n"
        f"imWidth={w}\n"
        f"imHeight={h}\n"
        f"imExt=.jpg\n"
    )
    with open(base / "gt" / "mot_challenge" / "MOT17-val" / seq_name / "seqinfo.ini", "w") as f:
        f.write(seqinfo)

    # Prediction folders
    pred_src = Path("predictions") / f"{seq_name}.txt"
    pred_dst = base / "trackers" / "mot_challenge" / "MOT17-val" / "my_tracker" / "data"
    pred_dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pred_src, pred_dst / f"{seq_name}.txt")

    print(f"  {seq_name}")

seqmap_dir = base / "gt" / "mot_challenge" / "seqmaps"
seqmap_dir.mkdir(parents=True, exist_ok=True)
with open(seqmap_dir / "MOT17-val.txt", "w") as f:
    f.write("name\n")
    for seq_dir in sorted(Path("val_data_withgt").iterdir()):
        if seq_dir.is_dir():
            f.write(f"{seq_dir.name}\n")

print("\n TrackEval folder structure ready")

  MOT17-02-DPM
  MOT17-04-DPM
  MOT17-05-DPM
  MOT17-09-DPM
  MOT17-10-DPM
  MOT17-11-DPM
  MOT17-13-DPM

 TrackEval folder structure ready


In [41]:
import cv2
from pathlib import Path

val_root = Path("val_data_withgt")
base     = Path("trackeval_data")

for seq_dir in sorted(val_root.iterdir()):
    if not seq_dir.is_dir():
        continue

    seq_name = seq_dir.name
    imgs     = sorted((seq_dir / "img1").glob("*.jpg"))
    img      = cv2.imread(str(imgs[0]))
    h, w     = img.shape[:2]

    seqinfo = (
        f"[Sequence]\n"
        f"name={seq_name}\n"
        f"imDir=img1\n"
        f"frameRate=30\n"
        f"seqLength={len(imgs)}\n"   
        f"imWidth={w}\n"
        f"imHeight={h}\n"
        f"imExt=.jpg\n"
    )

    ini_path = base / "gt" / "mot_challenge" / "MOT17-val" / seq_name / "seqinfo.ini"
    with open(ini_path, "w") as f:
        f.write(seqinfo)

    print(f"  {seq_name}  ({len(imgs)} frames, {w}x{h})")

print("\n All seqinfo.ini files fixed")

  MOT17-02-DPM  (114 frames, 1920x1080)
  MOT17-04-DPM  (199 frames, 1920x1080)
  MOT17-05-DPM  (154 frames, 640x480)
  MOT17-09-DPM  (103 frames, 1920x1080)
  MOT17-10-DPM  (138 frames, 1920x1080)
  MOT17-11-DPM  (197 frames, 1920x1080)
  MOT17-13-DPM  (159 frames, 1920x1080)

 All seqinfo.ini files fixed


In [43]:
from pathlib import Path

val_root  = Path("val_data_withgt")
base      = Path("trackeval_data")
pred_root = Path("predictions")

for seq_dir in sorted(val_root.iterdir()):
    if not seq_dir.is_dir():
        continue

    seq_name = seq_dir.name
    imgs     = sorted((seq_dir / "img1").glob("*.jpg"))

   
    frame_map = {int(p.stem): i+1 for i, p in enumerate(imgs)}
    print(f"{seq_name}  |  frames: {min(frame_map)} - {max(frame_map)}  →  1 - {len(imgs)}")

 
    gt_path = base / "gt" / "mot_challenge" / "MOT17-val" / seq_name / "gt" / "gt.txt"
    new_gt_lines = []
    with open(gt_path) as f:
        for line in f:
            parts = line.strip().split(",")
            orig_frame = int(parts[0])
            if orig_frame in frame_map:
                parts[0] = str(frame_map[orig_frame])
                new_gt_lines.append(",".join(parts) + "\n")

    with open(gt_path, "w") as f:
        f.writelines(new_gt_lines)
    print(f"  GT remapped:   {len(new_gt_lines)} rows")

    
    pred_path = base / "trackers" / "mot_challenge" / "MOT17-val" / "my_tracker" / "data" / f"{seq_name}.txt"
    new_pred_lines = []
    with open(pred_path) as f:
        for line in f:
            parts = line.strip().split(",")
            orig_frame = int(parts[0])
            if orig_frame in frame_map:
                parts[0] = str(frame_map[orig_frame])
                new_pred_lines.append(",".join(parts) + "\n")

    with open(pred_path, "w") as f:
        f.writelines(new_pred_lines)
    print(f"  Preds remapped: {len(new_pred_lines)} rows")

print("\n All frame numbers remapped to consecutive")

MOT17-02-DPM  |  frames: 7 - 599  →  1 - 114
  GT remapped:   2115 rows
  Preds remapped: 3938 rows
MOT17-04-DPM  |  frames: 2 - 1045  →  1 - 199
  GT remapped:   8532 rows
  Preds remapped: 9414 rows
MOT17-05-DPM  |  frames: 1 - 830  →  1 - 154
  GT remapped:   963 rows
  Preds remapped: 2450 rows
MOT17-09-DPM  |  frames: 2 - 512  →  1 - 103
  GT remapped:   809 rows
  Preds remapped: 1458 rows
MOT17-10-DPM  |  frames: 2 - 653  →  1 - 138
  GT remapped:   2333 rows
  Preds remapped: 5759 rows
MOT17-11-DPM  |  frames: 7 - 894  →  1 - 197
  GT remapped:   1806 rows
  Preds remapped: 3610 rows
MOT17-13-DPM  |  frames: 4 - 748  →  1 - 159
  GT remapped:   2231 rows
  Preds remapped: 7083 rows

 All frame numbers remapped to consecutive


In [44]:
import sys
sys.argv = ['']  

import trackeval

dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
dataset_config.update({
    'GT_FOLDER'        : 'trackeval_data/gt/mot_challenge',
    'TRACKERS_FOLDER'  : 'trackeval_data/trackers/mot_challenge',
    'BENCHMARK'        : 'MOT17',
    'SPLIT_TO_EVAL'    : 'val',
    'TRACKERS_TO_EVAL' : ['my_tracker'],
    'CLASSES_TO_EVAL'  : ['pedestrian'],
    'PRINT_RESULTS'    : True,
    'OUTPUT_FOLDER'    : 'trackeval_results',
})

eval_config = trackeval.Evaluator.get_default_eval_config()
eval_config.update({
    'DISPLAY_LESS_PROGRESS' : True,
    'PRINT_ONLY_COMBINED'   : True,
})

dataset    = trackeval.datasets.MotChallenge2DBox(dataset_config)
metrics    = [trackeval.metrics.HOTA(), 
              trackeval.metrics.CLEAR(), 
              trackeval.metrics.Identity()]
evaluator  = trackeval.Evaluator(eval_config)

results, _ = evaluator.evaluate([dataset], metrics)


MotChallenge2DBox Config:
GT_FOLDER            : trackeval_data/gt/mot_challenge
TRACKERS_FOLDER      : trackeval_data/trackers/mot_challenge
OUTPUT_FOLDER        : trackeval_results             
TRACKERS_TO_EVAL     : ['my_tracker']                
CLASSES_TO_EVAL      : ['pedestrian']                
BENCHMARK            : MOT17                         
SPLIT_TO_EVAL        : val                           
INPUT_AS_ZIP         : False                         
PRINT_CONFIG         : True                          
DO_PREPROC           : True                          
TRACKER_SUB_FOLDER   : data                          
OUTPUT_SUB_FOLDER    :                               
TRACKER_DISPLAY_NAMES : None                          
SEQMAP_FOLDER        : None                          
SEQMAP_FILE          : None                          
SEQ_INFO             : None                          
GT_LOC_FORMAT        : {gt_folder}/{seq}/gt/gt.txt   
SKIP_SPLIT_FOL       : False                  

<Figure size 640x480 with 0 Axes>

Improving the detections

In [45]:
import os
import cv2
import glob
import numpy as np
from pathlib import Path
from tqdm import tqdm
from deep_sort_realtime.deepsort_tracker import DeepSort

val_root  = Path("val_data_withgt")
pred_root = Path("predictions")
pred_root.mkdir(exist_ok=True)

for seq_dir in sorted(val_root.iterdir()):
    if not seq_dir.is_dir():
        continue

    seq_name  = seq_dir.name
    img_files = sorted((seq_dir / "img1").glob("*.jpg"))
    print(f"\n[{seq_name}] — {len(img_files)} frames")

    tracker = DeepSort(
        max_age=30,
        n_init=3,
        nn_budget=100,
        max_cosine_distance=0.3
    )

    pred_lines = []

    for img_path in tqdm(img_files, desc=f"  Tracking"):
        frame_num    = int(img_path.stem)
        frame_bgr    = cv2.imread(str(img_path))
        frame_rgb    = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = frame_rgb.shape[:2]
        frame_resized  = cv2.resize(frame_rgb, (640, 640))
        scale_x = orig_w / 640
        scale_y = orig_h / 640

        boxes, mean_scores, uncertainty = mc_dropout_inference(
            model, frame_resized, device,
            score_threshold=0.85,       # ← raised from 0.7
            n_runs=10
        )

        if len(boxes) > 0:
            boxes[:, [0, 2]] *= scale_x
            boxes[:, [1, 3]] *= scale_y

        embeddings = extract_reid_features(reid_model, frame_rgb, boxes, device)

        filtered_boxes, filtered_scores, filtered_embeddings, _ = uncertainty_gate(
            boxes, mean_scores, uncertainty, embeddings,
            uncertainty_threshold=0.03  # ← lowered from 0.05
        )

        deepsort_dets = format_for_deepsort(
            filtered_boxes, filtered_scores, filtered_embeddings,
            orig_h, orig_w
        )
        tracks = tracker.update_tracks(deepsort_dets, frame=frame_rgb)

        for t in tracks:
            if not t.is_confirmed():
                continue
            x1, y1, x2, y2 = t.to_ltrb()
            w = x2 - x1
            h = y2 - y1
            pred_lines.append(
                f"{frame_num},{t.track_id},{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},1,-1,-1,-1\n"
            )

    pred_file = pred_root / f"{seq_name}.txt"
    with open(pred_file, "w") as f:
        f.writelines(pred_lines)
    print(f"  Predictions saved: {len(pred_lines)} rows → {pred_file}")

print("\n All sequences tracked and saved")

/home/hardik/.local/lib/python3.8/site-packages/deep_sort_realtime/embedder/embedder_pytorch.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(


[MOT17-02-DPM] — 114 frames


  Tracking:   3%|▎         | 3/114 [00:02<01:49,  1.01it/s]

  Detection 21 filtered — uncertainty 0.0810 > 0.03
  Detection 22 filtered — uncertainty 0.1813 > 0.03
  Detection 23 filtered — uncertainty 0.1943 > 0.03


  Tracking:   4%|▎         | 4/114 [00:03<01:48,  1.01it/s]

...

  Detection 6 filtered — uncertainty 0.0823 > 0.03


  Tracking:  99%|█████████▉| 158/159 [02:11<00:00,  1.62it/s]

  Detection 5 filtered — uncertainty 0.1377 > 0.03


  Tracking: 100%|██████████| 159/159 [02:11<00:00,  1.21it/s]

  Predictions saved: 6418 rows → predictions/MOT17-13-DPM.txt

 All sequences tracked and saved


In [46]:
import shutil
from pathlib import Path

pred_root = Path("predictions")
pred_dst  = Path("trackeval_data/trackers/mot_challenge/MOT17-val/my_tracker/data")

for f in sorted(pred_root.glob("*.txt")):
    shutil.copy2(f, pred_dst / f.name)
    print(f"  Copied: {f.name}")

print("\n Predictions copied")

  Copied: MOT17-02-DPM.txt
  Copied: MOT17-04-DPM.txt
  Copied: MOT17-05-DPM.txt
  Copied: MOT17-09-DPM.txt
  Copied: MOT17-10-DPM.txt
  Copied: MOT17-11-DPM.txt
  Copied: MOT17-13-DPM.txt

 Predictions copied


In [47]:
from pathlib import Path

val_root  = Path("val_data_withgt")
base      = Path("trackeval_data")

for seq_dir in sorted(val_root.iterdir()):
    if not seq_dir.is_dir():
        continue

    seq_name  = seq_dir.name
    imgs      = sorted((seq_dir / "img1").glob("*.jpg"))
    frame_map = {int(p.stem): i+1 for i, p in enumerate(imgs)}

    pred_path = base / "trackers" / "mot_challenge" / "MOT17-val" / "my_tracker" / "data" / f"{seq_name}.txt"
    new_lines = []
    with open(pred_path) as f:
        for line in f:
            parts      = line.strip().split(",")
            orig_frame = int(parts[0])
            if orig_frame in frame_map:
                parts[0] = str(frame_map[orig_frame])
                new_lines.append(",".join(parts) + "\n")

    with open(pred_path, "w") as f:
        f.writelines(new_lines)
    print(f"  {seq_name}  |  remapped: {len(new_lines)} rows")

print("\n Frame numbers remapped")

  MOT17-02-DPM  |  remapped: 3947 rows
  MOT17-04-DPM  |  remapped: 9382 rows
  MOT17-05-DPM  |  remapped: 2260 rows
  MOT17-09-DPM  |  remapped: 1533 rows
  MOT17-10-DPM  |  remapped: 5450 rows
  MOT17-11-DPM  |  remapped: 3697 rows
  MOT17-13-DPM  |  remapped: 6418 rows

 Frame numbers remapped


In [48]:
import sys
sys.argv = ['']
import trackeval

dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
dataset_config.update({
    'GT_FOLDER'        : 'trackeval_data/gt/mot_challenge',
    'TRACKERS_FOLDER'  : 'trackeval_data/trackers/mot_challenge',
    'BENCHMARK'        : 'MOT17',
    'SPLIT_TO_EVAL'    : 'val',
    'TRACKERS_TO_EVAL' : ['my_tracker'],
    'CLASSES_TO_EVAL'  : ['pedestrian'],
    'PRINT_RESULTS'    : True,
    'OUTPUT_FOLDER'    : 'trackeval_results',
})

eval_config = trackeval.Evaluator.get_default_eval_config()
eval_config.update({
    'DISPLAY_LESS_PROGRESS' : True,
    'PRINT_ONLY_COMBINED'   : True,
})

dataset   = trackeval.datasets.MotChallenge2DBox(dataset_config)
metrics   = [trackeval.metrics.HOTA(),
             trackeval.metrics.CLEAR(),
             trackeval.metrics.Identity()]
evaluator = trackeval.Evaluator(eval_config)

results, _ = evaluator.evaluate([dataset], metrics)


MotChallenge2DBox Config:
GT_FOLDER            : trackeval_data/gt/mot_challenge
TRACKERS_FOLDER      : trackeval_data/trackers/mot_challenge
OUTPUT_FOLDER        : trackeval_results             
TRACKERS_TO_EVAL     : ['my_tracker']                
CLASSES_TO_EVAL      : ['pedestrian']                
BENCHMARK            : MOT17                         
SPLIT_TO_EVAL        : val                           
INPUT_AS_ZIP         : False                         
PRINT_CONFIG         : True                          
DO_PREPROC           : True                          
TRACKER_SUB_FOLDER   : data                          
OUTPUT_SUB_FOLDER    :                               
TRACKER_DISPLAY_NAMES : None                          
SEQMAP_FOLDER        : None                          
SEQMAP_FILE          : None                          
SEQ_INFO             : None                          
GT_LOC_FORMAT        : {gt_folder}/{seq}/gt/gt.txt   
SKIP_SPLIT_FOL       : False                  

<Figure size 640x480 with 0 Axes>